In [2]:
import pandas as pd 
df=pd.read_csv("../data/IMDB_dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [11]:
df.shape

df["sentiment"].value_counts()


sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [15]:
print("Before removing duplicates:", df.shape)

df = df.drop_duplicates()

print("After removing duplicates:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

Before removing duplicates: (50000, 2)
After removing duplicates: (49582, 2)
Duplicate rows: 0


In [17]:
print(df["sentiment"].value_counts())
print(df["sentiment"].value_counts(normalize=True)*100)

sentiment
positive    24884
negative    24698
Name: count, dtype: int64
sentiment
positive    50.187568
negative    49.812432
Name: proportion, dtype: float64


In [18]:
print(df.columns.to_list())

['review', 'sentiment']


In [21]:
x=df["review"]
y=df["sentiment"]

x.shape
y.shape


(49582,)

In [26]:
from sklearn.model_selection import train_test_split


x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2,stratify=y)

print(len(x_train))
print(len(x_test))

print(y_train.value_counts(normalize=True)*100)
print(y_test.value_counts(normalize=True)*100)

39665
9917
sentiment
positive    50.187823
negative    49.812177
Name: proportion, dtype: float64
sentiment
positive    50.186548
negative    49.813452
Name: proportion, dtype: float64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

sent_tfidf=TfidfVectorizer(
    lowercase=True,
    sublinear_tf=True,
    ngram_range=(1, 2),
    max_features=10000
)
x_train_tfidf = sent_tfidf.fit_transform(x_train)
x_test_tfidf = sent_tfidf.transform(x_test)

print("Training shape:", x_train_tfidf.shape)
print("Testing shape:", x_test_tfidf.shape)

Training shape: (39665, 10000)
Testing shape: (9917, 10000)


In [ ]:
from sklearn.svm import LinearSVC

sent_model=LinearSVC(C=1.0)

sent_model.fit(x_train_tfidf,y_train)
print("Sentiment model training completed.")

Sentiment model training completed.


In [35]:
y_pred_sent=sent_model.predict(x_test_tfidf)
print(y_pred_sent[:10])

['negative' 'positive' 'positive' 'negative' 'negative' 'positive'
 'negative' 'negative' 'positive' 'negative']


In [38]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred_sent)

print("Sentiment Model Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_sent,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_sent))

Sentiment Model Accuracy: 0.8845416960774428

Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.88      0.88      4940
    Positive       0.88      0.89      0.89      4977

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917


Confusion Matrix:
[[4338  602]
 [ 543 4434]]


In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sent_lr = LogisticRegression(max_iter=1000)

sent_lr.fit(x_train_tfidf, y_train)

y_pred_lr = sent_lr.predict(x_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8911969345568216

Classification Report:
              precision    recall  f1-score   support

    Negative       0.90      0.88      0.89      4940
    Positive       0.88      0.91      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917


Confusion Matrix:
[[4327  613]
 [ 466 4511]]
